# 🤖 Huấn Luyện Mô Hình
## Mô hình Đánh giá Rủi ro Tín dụng
**Nhóm 14 - Bài tập lớn môn Học Máy**

---

### Các mô hình sử dụng
1. Logistic Regression
2. Decision Tree
3. Random Forest
4. Gradient Boosting
5. XGBoost
6. LightGBM
7. SVM
8. KNN
9. Stacking Ensemble

## 1. Chuẩn bị

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from src.data.load_data import load_german_credit, load_give_me_credit
from src.data.preprocess import preprocess_pipeline
from src.features.feature_engineering import (
    create_credit_features_german, create_credit_features_gmsc,
    feature_engineering_pipeline, get_feature_importance
)
from src.models.train_model import (
    get_default_models, train_all_models, cross_validate_models,
    tune_all_models, create_stacking_model, save_all_models
)

print('✅ Import thành công!')

## 2. Tiền xử lý dữ liệu

In [ ]:
# Nạp và tiền xử lý German Credit
df_german = load_german_credit()
df_german = create_credit_features_german(df_german)

prep_german = preprocess_pipeline(
    df_german, target_col='target',
    encoding_method='label', scaling_method='standard',
    imbalance_method='smote'
)

X_train_g, X_test_g = prep_german['X_train'], prep_german['X_test']
y_train_g, y_test_g = prep_german['y_train'], prep_german['y_test']
print(f'German Credit - Train: {X_train_g.shape}, Test: {X_test_g.shape}')

In [ ]:
# Nạp và tiền xử lý Give Me Some Credit
df_gmsc = load_give_me_credit()
df_gmsc = create_credit_features_gmsc(df_gmsc)

prep_gmsc = preprocess_pipeline(
    df_gmsc, target_col='SeriousDlqin2yrs',
    encoding_method='label', scaling_method='standard',
    imbalance_method='smote'
)

X_train_gm, X_test_gm = prep_gmsc['X_train'], prep_gmsc['X_test']
y_train_gm, y_test_gm = prep_gmsc['y_train'], prep_gmsc['y_test']
print(f'Give Me Some Credit - Train: {X_train_gm.shape}, Test: {X_test_gm.shape}')

## 3. Cross-Validation

In [ ]:
# Cross-validation trên German Credit
print('=' * 60)
print('CROSS-VALIDATION - GERMAN CREDIT')
print('=' * 60)
cv_german = cross_validate_models(X_train_g, y_train_g, cv=5, scoring='f1')
display(cv_german)

In [ ]:
# Cross-validation trên Give Me Some Credit
print('=' * 60)
print('CROSS-VALIDATION - GIVE ME SOME CREDIT')
print('=' * 60)
cv_gmsc = cross_validate_models(X_train_gm, y_train_gm, cv=5, scoring='f1')
display(cv_gmsc)

## 4. Huấn luyện mô hình

In [ ]:
# Huấn luyện trên German Credit
print('HUẤN LUYỆN - GERMAN CREDIT')
trained_german = train_all_models(X_train_g, y_train_g)

# Thêm Stacking
stacking_g = create_stacking_model()
stacking_g.fit(X_train_g, y_train_g)
trained_german['Stacking Ensemble'] = {'model': stacking_g}

In [ ]:
# Huấn luyện trên Give Me Some Credit
print('HUẤN LUYỆN - GIVE ME SOME CREDIT')
trained_gmsc = train_all_models(X_train_gm, y_train_gm)

# Thêm Stacking
stacking_gm = create_stacking_model()
stacking_gm.fit(X_train_gm, y_train_gm)
trained_gmsc['Stacking Ensemble'] = {'model': stacking_gm}

## 5. Hyperparameter Tuning

Sử dụng **RandomizedSearchCV** để tìm siêu tham số tối ưu.

In [ ]:
# Hyperparameter Tuning cho German Credit (có thể tốn thời gian)
# Bỏ comment để chạy:
# tuned_german = tune_all_models(X_train_g, y_train_g, method='random', n_iter=20)
# for name, result in tuned_german.items():
#     print(f'{name}: Best F1 = {result["best_score"]:.4f}')
#     print(f'  Params: {result["best_params"]}')

## 6. Lưu mô hình

In [ ]:
# Lưu tất cả mô hình
paths_german = save_all_models(trained_german)
print('\nĐã lưu mô hình German Credit:')
for name, path in paths_german.items():
    print(f'  {name}: {path}')

paths_gmsc = save_all_models(trained_gmsc)
print('\nĐã lưu mô hình Give Me Some Credit:')
for name, path in paths_gmsc.items():
    print(f'  {name}: {path}')

## Tổng kết

- Đã huấn luyện 9 mô hình ML trên 2 bộ dữ liệu
- Cross-Validation cho kết quả ổn định
- Stacking Ensemble kết hợp sức mạnh nhiều mô hình
- Tiếp theo: Đánh giá chi tiết trong notebook 03_Evaluation